# Error handling and file handling

## Learning objectives

By the end of this notebook you will be able to:

- read error messages and recognise common error types
- handle errors with try, except, else and finally
- raise errors for invalid input
- read and write text files safely with with open()
- read and write CSV files with the csv module
- handle missing files and invalid values

## Introduction

Errors are part of programming: a file is missing, a value is text instead of a number, a list is empty.
Good analysis code expects these situations and handles them clearly instead of crashing halfway or,
worse, carrying on with wrong values. This notebook also covers reading and writing files, where many
errors happen.

## Reading an error message

Python prints a *traceback*. Read the **last line first**: it names the error type and gives a message.
The lines above show where it happened.

| error | typical cause |
|---|---|
| `NameError` | a name that does not exist (typo, or cell not run) |
| `TypeError` | an operation on the wrong type, e.g. `"5" + 3` |
| `ValueError` | right type, impossible value, e.g. `int("five")` |
| `KeyError` | a dictionary key that does not exist |
| `IndexError` | a list position that does not exist |
| `ZeroDivisionError` | dividing by zero |
| `FileNotFoundError` | a file path that does not exist |
| `IndentationError`, `SyntaxError` | code Python cannot read |

In [1]:
for code in ['int("five")', '"5" + 3', '[1, 2][5]', '{"a": 1}["b"]', '1 / 0']:
    try:
        eval(code)
    except Exception as e:
        print(f"{code:16} -> {type(e).__name__}: {e}")

int("five")      -> ValueError: invalid literal for int() with base 10: 'five'
"5" + 3          -> TypeError: can only concatenate str (not "int") to str
[1, 2][5]        -> IndexError: list index out of range
{"a": 1}["b"]    -> KeyError: 'b'
1 / 0            -> ZeroDivisionError: division by zero


## try and except

Put code that might fail in `try`. If an error of the named type happens, the `except` block runs
instead of the program stopping.

In [2]:
def to_number(text):
    try:
        return float(text)
    except ValueError:
        return None

values = ["12.5", "n/a", "7", "", "3.2e1"]
print([to_number(v) for v in values])

[12.5, None, 7.0, None, 32.0]


Catch the **specific** error you expect. A bare `except:` also catches typos and other bugs, hiding them.
Several types can share one block:

In [3]:
def safe_ratio(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float("nan")
    except TypeError:
        return None

print(safe_ratio(10, 4), safe_ratio(10, 0), safe_ratio("10", 4))

2.5 nan None


## else and finally

- `else` runs only if the `try` block succeeded
- `finally` always runs, error or not, which is where clean-up goes

In [4]:
def parse_age(text):
    try:
        age = int(text)
    except ValueError:
        print(f"  {text!r} is not a whole number")
        return None
    else:
        print(f"  parsed {age}")
        return age
    finally:
        print("  (finished checking", repr(text) + ")")

parse_age("42")
parse_age("forty")

  parsed 42
  (finished checking '42')
  'forty' is not a whole number
  (finished checking 'forty')


## Raising errors

When input does not make sense, stop with a clear message rather than returning a wrong answer.

In [5]:
def bmi(weight_kg, height_m):
    if height_m <= 0 or weight_kg <= 0:
        raise ValueError(f"weight and height must be positive, got {weight_kg} and {height_m}")
    if height_m > 3:
        raise ValueError(f"height {height_m} looks like centimetres; give metres")
    return weight_kg / height_m ** 2

for w, h in [(70, 1.75), (70, 175), (70, 0)]:
    try:
        print(round(bmi(w, h), 1))
    except ValueError as e:
        print("ValueError:", e)

22.9
ValueError: height 175 looks like centimetres; give metres
ValueError: weight and height must be positive, got 70 and 0


## File handling

### Opening files with with

`with open(...) as f:` opens a file and closes it automatically when the block ends, even if an error
happens.

| mode | meaning |
|---|---|
| `"r"` | read (the default) |
| `"w"` | write, replacing the file if it exists |
| `"a"` | append to the end |
| `"x"` | create; fail if the file exists |

Always give `encoding="utf-8"` for text files so special characters work on every computer.

In [6]:
from pathlib import Path
work = Path("scratch")
work.mkdir(exist_ok=True)

notes = work / "notes.txt"
with open(notes, "w", encoding="utf-8") as f:
    f.write("Run 1: 48 samples processed\n")
    f.write("Run 2: 52 samples processed\n")

with open(notes, "a", encoding="utf-8") as f:
    f.write("Run 3: 47 samples processed\n")

with open(notes, encoding="utf-8") as f:
    content = f.read()
print(content)

Run 1: 48 samples processed
Run 2: 52 samples processed
Run 3: 47 samples processed



Reading line by line uses little memory, even for huge files:

In [7]:
total = 0
with open(notes, encoding="utf-8") as f:
    for line in f:
        number = int(line.split(":")[1].split()[0])
        total += number
print("total samples:", total)

total samples: 147


`pathlib` has shortcuts for small files:

In [8]:
print(notes.read_text(encoding="utf-8").splitlines())

['Run 1: 48 samples processed', 'Run 2: 52 samples processed', 'Run 3: 47 samples processed']


### Reading CSV files with the csv module

CSV (comma-separated values) is the most common data file format. `csv.DictReader` reads each row as a
dictionary keyed by the column names. Note that **every value arrives as text**.

In [9]:
import csv

with open("../datasets/raw/patients.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(len(rows), "rows; columns:", list(rows[0].keys()))
print(rows[0])
print(type(rows[0]["age"]))

300 rows; columns: ['patient_id', 'site', 'sex', 'age', 'weight_kg', 'height_cm', 'smoker', 'group', 'enrolled']
{'patient_id': 'P0001', 'site': 'Site A', 'sex': 'M', 'age': '41', 'weight_kg': '66.4', 'height_cm': '156.3', 'smoker': 'No', 'group': 'Treatment', 'enrolled': '2025-03-31'}
<class 'str'>


In [10]:
ages = [int(r["age"]) for r in rows]
weights = [float(r["weight_kg"]) for r in rows if r["weight_kg"] != ""]
print(f"mean age {sum(ages) / len(ages):.1f}; mean weight {sum(weights) / len(weights):.1f} "
      f"({len(rows) - len(weights)} weights missing)")

mean age 51.4; mean weight 70.0 (10 weights missing)


### Writing CSV files

In [11]:
summary = {}
for r in rows:
    summary.setdefault(r["site"], []).append(int(r["age"]))

out = work / "age_by_site.csv"
with open(out, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["site", "n", "mean_age"])
    for site, a in sorted(summary.items()):
        writer.writerow([site, len(a), round(sum(a) / len(a), 1)])

print(out.read_text())

site,n,mean_age
Site A,111,49.6
Site B,87,56.0
Site C,54,50.8
Site D,48,48.2



`newline=""` is required by the `csv` module on all platforms to avoid blank lines on Windows.

In practice, pandas reads and writes CSV files in one line (module 6), but the `csv` module is useful
for small scripts and for files that are not quite tables.

### Handling missing files

In [12]:
def load_lines(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read().splitlines()
    except FileNotFoundError:
        print(f"File not found: {path}. Check the name and the folder you are working in.")
        return []

print(load_lines(notes)[:1])
print(load_lines(work / "missing.txt"))

['Run 1: 48 samples processed']
File not found: scratch/missing.txt. Check the name and the folder you are working in.
[]


### Handling invalid values while reading

A robust reader keeps going past bad rows and reports them:

In [13]:
bad_file = work / "readings.csv"
bad_file.write_text("id,value\nA,5.4\nB,n/a\nC,7.1\nD,\nE,6.8\n")

good, problems = [], []
with open(bad_file, encoding="utf-8", newline="") as f:
    for line_no, row in enumerate(csv.DictReader(f), start=2):
        try:
            good.append((row["id"], float(row["value"])))
        except ValueError:
            problems.append((line_no, row["id"], row["value"]))

print("valid:", good)
print("problems (line, id, value):", problems)

valid: [('A', 5.4), ('C', 7.1), ('E', 6.8)]
problems (line, id, value): [(3, 'B', 'n/a'), (5, 'D', '')]


In [14]:
import shutil
shutil.rmtree(work)          # remove the demo folder

## Practise

Exercises for this folder are in `exercises/Functions_and_Modules_Exercises.ipynb`.

## Further reading

- [Python tutorial: errors and exceptions](https://docs.python.org/3/tutorial/errors.html)
- [Built-in exceptions](https://docs.python.org/3/library/exceptions.html)
- [Reading and writing files](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)
- [csv module](https://docs.python.org/3/library/csv.html)
- [pathlib](https://docs.python.org/3/library/pathlib.html)